In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MY_TOKEN = ""
# REPO_ID = "username/private-repo-name"

# login(token=MY_TOKEN)

# tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
# model = AutoModelForSequenceClassification.from_pretrained(REPO_ID)

In [ ]:
model_path = 'ssurface/dimabsa-2026-subtask1-private-mmbert-alldata_last'
test_file = '/kaggle/working/dimabsa/dimabsa_test_gold'
output_file = 'predictions/mmbert-alldata_last'
batch_size = 32
max_len = 50
model_name = 'jhu-clsp/mmBERT-base'

In [ ]:
import argparse
import os
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification ,AutoModelForCausalLM
from tqdm import tqdm
from dataloader import Dataloader


def gen_sub(model_path,model_name, batch_size, max_len ,output_file ,test_data):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    model.to(device)
    model.eval()

    # Accept either a single file or a directory of test files
    files_to_process = []
    if test_data and os.path.isdir(test_data):
        for p in sorted(os.listdir(test_data)):
            if p.lower().endswith('.jsonl') and 'test_task1' in p.lower():
                files_to_process.append(os.path.join(test_data, p))
    elif test_data:
        files_to_process = [test_data]

    if not files_to_process:
        raise SystemExit(f"No test files found at {test_data}")

    # If processing multiple files, output_file should be treated as a directory
    if os.path.isdir(test_data):
        os.makedirs(output_file, exist_ok=True)
    else:
        os.makedirs(os.path.dirname(output_file) or '.', exist_ok=True)

    for file_path in files_to_process:
        print(f"Processing {file_path} ...")
        dataset_list = Dataloader._parse_jsonl(file_path)
        
        # Pass the robust tokenizer object
        dataset = Dataloader(dataset_list, model_name , max_len=max_len)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        raw_data = dataset.data
        results = []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Inferring {os.path.basename(file_path)}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                token_type_ids = batch.get('token_type_ids', None)

                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
                else:
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

                preds = outputs.logits.cpu().numpy()
                results.extend(preds)

        if len(results) != len(raw_data):
            print(f"Warning: predictions ({len(results)}) != examples ({len(raw_data)}). Truncating to min length.")
        n = min(len(results), len(raw_data))

        submission_map = {}
        csv_data = []

        for i in range(n):
            row = raw_data[i]
            val_pred = float(results[i][0])
            aro_pred = float(results[i][1])
            val_pred = (val_pred * 8.0) + 1.0
            aro_pred = (aro_pred * 8.0) + 1.0
            val_str = f"{val_pred:.2f}"
            aro_str = f"{aro_pred:.2f}"

            doc_id = row.get('ID', f'unk_{i}')
            target = row.get('Target', 'general')

            if doc_id not in submission_map:
                submission_map[doc_id] = {
                    "ID": doc_id,
                    "Aspect_VA": []
                }

            submission_map[doc_id]["Aspect_VA"].append({
                "Aspect": target,
                "VA": f"{val_str}#{aro_str}"
            })

            csv_data.append({
                "ID": doc_id,
                "Target": target,
                "Valence": val_pred,
                "Arousal": aro_pred
            })

        final_json = list(submission_map.values())

        out_json = output_file
        if os.path.isdir(test_data):
            base = os.path.splitext(os.path.basename(file_path))[0]
            model_tag = os.path.basename(model_path).replace('/', '-')
            # Save INSIDE the directory specified by output_file
            out_json = os.path.join(output_file, f"{model_tag}_{base}.jsonl")
        elif not out_json.endswith('.jsonl') and out_json.endswith('.json'):
             out_json = out_json.replace('.json', '.jsonl')

        # Write as JSONL
        with open(out_json, 'w', encoding='utf-8') as f:
            for entry in final_json:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')

        csv_filename = out_json.replace(".jsonl", ".csv").replace(".json", ".csv")

In [ ]:
import argparse
import os
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from dataloader import Dataloader
from scipy.special import expit
from functools import partial

tqdm.tqdm = partial(tqdm.tqdm, gui=False, display=True)
def gen_sub(model_path, model_name, batch_size, max_len, output_file, test_data):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    files_to_process = []
    if os.path.isdir(test_data):
        for p in sorted(os.listdir(test_data)):
            if p.lower().endswith('.jsonl'):
                files_to_process.append(os.path.join(test_data, p))
        os.makedirs(output_file, exist_ok=True)
    else:
        files_to_process = [test_data]
        os.makedirs(os.path.dirname(output_file) or '.', exist_ok=True)

    if not files_to_process:
        raise SystemExit("No test files found")

    for file_path in files_to_process:
        print(f"Processing {file_path}")
        
        filename = os.path.basename(file_path).lower()
        if 'laptop' in filename:
            domain_label = "laptop"
        elif 'restaurant' in filename:
            domain_label = "restaurant"
        elif 'finance' in filename:
            domain_label = "finance"
        elif 'hotel' in filename:
            domain_label = "hotel"
        else:
            domain_label = "general"

        dataset_list = Dataloader._parse_jsonl(file_path)
        
        for item in dataset_list:
            original_text = item.get('Text') or item.get('Sentence') or ""
            item['Text'] = "Domain:" + domain_label + "Text:" + str(original_text)

        dataset = Dataloader(dataset_list, model_name, max_len=max_len)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        raw_data = dataset.data
        results = []

        with torch.no_grad():
            for batch in tqdm(loader):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                token_type_ids = batch.get('token_type_ids', None)

                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
                else:
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

                logits = outputs.logits.cpu().numpy()
                results.extend(logits)

        n = len(raw_data)
        submission_map = {}
        csv_data = []

        for i in range(n):
            row = raw_data[i]
            
            val_logit = float(results[i][0])
            aro_logit = float(results[i][1])
            
            val_pred = (expit(val_logit) * 8.0) + 1.0
            aro_pred = (expit(aro_logit) * 8.0) + 1.0
            
            val_pred = np.clip(val_pred, 1.0, 9.0)
            aro_pred = np.clip(aro_pred, 1.0, 9.0)

            val_str = f"{val_pred:.2f}"
            aro_str = f"{aro_pred:.2f}"

            doc_id = row.get('ID')
            target = row.get('Target')

            if doc_id not in submission_map:
                submission_map[doc_id] = {
                    "ID": doc_id,
                    "Aspect_VA": []
                }

            submission_map[doc_id]["Aspect_VA"].append({
                "Aspect": target,
                "VA": f"{val_str}#{aro_str}"
            })

            csv_data.append({
                "ID": doc_id,
                "Target": target,
                "Valence": val_pred,
                "Arousal": aro_pred
            })

        final_json = list(submission_map.values())

        if os.path.isdir(test_data):
            base = os.path.splitext(os.path.basename(file_path))[0]
            clean_base = base.replace('_test_task1', '').replace('_dev_task1', '')
            out_json = os.path.join(output_file, f"pred_{clean_base}.jsonl")
        else:
            out_json = output_file
            if not out_json.endswith('.jsonl'):
                out_json = out_json.replace('.json', '.jsonl')

        with open(out_json, 'w', encoding='utf-8') as f:
            for entry in final_json:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')

        csv_filename = out_json.replace(".jsonl", ".csv")
        pd.DataFrame(csv_data).to_csv(csv_filename, index=False)

In [ ]:
gen_sub( model_path ,  model_name, 32, max_len , output_file, test_file)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

2026-02-12 09:27:00.662938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770888420.942756    1210 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770888421.011287    1210 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770888421.591638    1210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770888421.591670    1210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770888421.591674    1210 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

Processing /kaggle/working/dimabsa/dimabsa_test_gold


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/dimabsa/dimabsa_test_gold'

In [ ]:
print('pred_dir:' , "/kaggle/working/dimabsa/bertcls_new/predictions/mmbert-cls-alldata" ,  "gold_labels:" , "/kaggle/working/dimabsa/dimabsa_test_gold")

pred_dir: /kaggle/working/dimabsa/bertcls_new/predictions/mmbert-cls-alldata gold_labels: /kaggle/working/dimabsa/dimabsa_test_gold


In [ ]:
import json
import os
import glob
import math
import argparse
from scipy.stats import pearsonr

def load_json_data(path):
    data = {}
    if not os.path.exists(path):
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            eid = entry["ID"]
            aspects = {}
            if "Aspect_VA" in entry:
                items = entry["Aspect_VA"]
            elif "Quadruplet" in entry:
                items = entry["Quadruplet"]
            elif "Aspect" in entry and "Intensity" in entry:
                items = []
                asps = entry["Aspect"]
                ints = entry["Intensity"]
                cats = entry.get("Category", [])
                for i in range(len(asps)):
                    a_name = asps[i]
                    if a_name == "NULL" or a_name is None:
                        a_name = cats[i].replace("#", " ") if i < len(cats) else "general"
                    items.append({"Aspect": a_name, "VA": ints[i]})
            else:
                items = []
            for item in items:
                asp_name = item["Aspect"]
                if asp_name == "NULL":
                    asp_name = item.get("Category", "general").replace("#", " ")
                aspects[asp_name.replace(" ", "").lower()] = item["VA"]
            data[eid] = aspects
    return data

def compare_metrics(gold, pred):
    gold_dir = gold
    pred_dir = pred
    pred_files = glob.glob(os.path.join(pred_dir, "*.jsonl"))
    all_results = []

    print(f"{'FILE':<35} | {'PCC_V':<8} | {'PCC_A':<8} | {'RMSE_VA':<8}")
    print("-" * 70)

    for p_path in sorted(pred_files):
        fname = os.path.basename(p_path)
        clean_name = fname.replace("pred_", "").replace(".jsonl", "")
        g_candidates = glob.glob(os.path.join(gold_dir, f"*{clean_name}*"))
        
        if not g_candidates:
            continue
            
        g_data = load_json_data(g_candidates[0])
        p_data = load_json_data(p_path)

        gv, ga, pv, pa = [], [], [], []
        for eid, g_aspects in g_data.items():
            if eid in p_data:
                p_aspects = p_data[eid]
                for asp_key, g_va in g_aspects.items():
                    if asp_key in p_aspects:
                        p_va = p_aspects[asp_key]
                        g_split = g_va.split("#")
                        p_split = p_va.split("#")
                        gv.append(float(g_split[0]))
                        ga.append(float(g_split[1]))
                        pv.append(float(p_split[0]))
                        pa.append(float(p_split[1]))

        if len(gv) > 1:
            res_v = pearsonr(gv, pv)[0]
            res_a = pearsonr(ga, pa)[0]
            
            diff_sq = []
            for i in range(len(gv)):
                diff_sq.append((gv[i] - pv[i])**2)
                diff_sq.append((ga[i] - pa[i])**2)
            
            rmse = math.sqrt(sum(diff_sq) / len(gv))
            
            all_results.append({"v": res_v, "a": res_a, "r": rmse})
            print(f"{fname[:35]:<35} | {res_v:.4f}   | {res_a:.4f}   | {rmse:.4f}")

    if all_results:
        avg_v = sum(x["v"] for x in all_results) / len(all_results)
        avg_a = sum(x["a"] for x in all_results) / len(all_results)
        avg_r = sum(x["r"] for x in all_results) / len(all_results)
        print("-" * 70)
        print(f"{'AVERAGE':<35} | {avg_v:.4f}   | {avg_a:.4f}   | {avg_r:.4f}")

In [ ]:
pred = "/kaggle/working/dimabsa/bertcls_new/predictions/mmbert-alldata_last"
gold = "/kaggle/working/dimabsa/dimabsa_test_gold"

In [ ]:
compare_metrics(pred, gold)

FILE                                | PCC_V    | PCC_A    | RMSE_VA 
----------------------------------------------------------------------
